## Make decoding movies

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pickle
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
# ignore datajoint+jupyter async warnings
import warnings
warnings.simplefilter('ignore', category=DeprecationWarning)
warnings.simplefilter('ignore', category=ResourceWarning)

In [3]:
from spyglass.shijiegu.Analysis_SGU import TrialChoice, DecodeIngredients, DecodeResults2D, ChangeofMindTheta
from spyglass.shijiegu.decodeHelpers import runSessionNames
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename
from spyglass.decoding.v0.visualization import make_single_environment_movie

[2025-10-07 23:02:24,624][INFO]: DataJoint 0.14.4 connected to shijiegu-alt@lmf-db.cin.ucsf.edu:3306


In [4]:
from spyglass.shijiegu.Analysis_SGU import RippleTimesWithDecode

### nwb file

In [5]:
nwb_copy_file_name = 'eliot20221021_.nwb'#'lewis20240109_.nwb'

In [7]:
session_interval, position_interval = runSessionNames(nwb_copy_file_name)

### session and time

In [8]:
TrialChoice & {'nwb_file_name':nwb_copy_file_name}

nwb_file_name name of the NWB file,epoch the session epoch for this task and apparatus(1 based),"epoch_name session name, get from IntervalList","choice_reward pandas dataframe, choice"
eliot20221021_.nwb,2,02_Seq2Session1,=BLOB=
eliot20221021_.nwb,4,04_Seq2Session2,=BLOB=
eliot20221021_.nwb,6,06_Seq2Session3,=BLOB=
eliot20221021_.nwb,8,08_Seq2Session4,=BLOB=


In [9]:
epoch_num = 8;

In [10]:
key={'nwb_file_name':nwb_copy_file_name,'epoch':epoch_num}
print(ChangeofMindTheta & key)
log = ChangeofMindTheta().fetch1_dataframe(key)

session_name = (TrialChoice & key).fetch1('epoch_name')

print(f"Session {session_name}")

[2025-10-07 22:55:51,983][WARNING]: Skipped checksum for file with hash: 8d8aef54-2885-a13c-2153-e9b42d84adc3, and path: /stelmo/nwb/analysis/eliot20221021/eliot20221021_R3YLLIK9U7.nwb


*nwb_file_name *epoch    *proportion    *delta_t_minus *delta_t_plus  *max_flag    analysis_file_
+------------+ +-------+ +------------+ +------------+ +------------+ +----------+ +------------+
eliot20221021_ 8         0.1            0              2              0            eliot20221021_
 (Total: 1)

Session 08_Seq2Session4


In [11]:
RippleTimesWithDecode & key

nwb_file_name name of the NWB file,interval_list_name descriptive name of this interval list,classifier_param_name a name for this set of parameters,encoding_set a name for this set of encoding,decode_threshold_method a name for this thresholding method,ripple_times ripple times within that interval
eliot20221021_.nwb,02_Seq2Session1,default_decoding_gpu_4armMaze,2Dheadspeed_above_4,MUA_05SD,=BLOB=
eliot20221021_.nwb,02_Seq2Session1,default_decoding_gpu_4armMaze,2Dheadspeed_above_4_andlowmua,MUA_05SD,=BLOB=
eliot20221021_.nwb,02_Seq2Session1,default_decoding_gpu_4armMaze,2Dheadspeed_above_4_andlowmua,MUA_0SD,=BLOB=
eliot20221021_.nwb,04_Seq2Session2,default_decoding_gpu_4armMaze,2Dheadspeed_above_4_andlowmua,MUA_05SD,=BLOB=
eliot20221021_.nwb,04_Seq2Session2,default_decoding_gpu_4armMaze,2Dheadspeed_above_4_andlowmua,MUA_0SD,=BLOB=
eliot20221021_.nwb,06_Seq2Session3,default_decoding_gpu_4armMaze,2Dheadspeed_above_4_andlowmua,MUA_05SD,=BLOB=
eliot20221021_.nwb,06_Seq2Session3,default_decoding_gpu_4armMaze,2Dheadspeed_above_4_andlowmua,MUA_0SD,=BLOB=
eliot20221021_.nwb,08_Seq2Session4,default_decoding_gpu_4armMaze,2Dheadspeed_above_4_andlowmua,MUA_05SD,=BLOB=
eliot20221021_.nwb,08_Seq2Session4,default_decoding_gpu_4armMaze,2Dheadspeed_above_4_andlowmua,MUA_0SD,=BLOB=


In [12]:
key={'nwb_file_name':nwb_copy_file_name, "interval_list_name":session_name,"decode_threshold_method":"MUA_0SD"}
ripple_log = pd.read_pickle((RippleTimesWithDecode & key).fetch1("ripple_times"))

In [13]:
ripple_log.loc[21]

start_time                                           1666397812.120548
end_time                                             1666397812.694548
animal_location                                                   home
trial_number                                                         7
cont_intvl           [[1666397812.121585, 1666397812.215585], [1666...
frag_intvl           [[1666397812.2175848, 1666397812.241585], [166...
cont_intvl_replay                                  [[], [0], [3], [0]]
duration                                                         0.574
mean_zscore                                                    2.00693
median_zscore                                                 1.253664
max_zscore                                                   25.086136
min_zscore                                                   -1.173996
max_speed                                                     1.534109
min_speed                                                     1.224652
median

In [14]:
entry = DecodeIngredients & {'nwb_file_name':nwb_copy_file_name,
                             'interval_list_name':session_name}
    
# Get data
marks = xr.open_dataset(entry.fetch1('marks'))
position_1d = pd.read_csv(entry.fetch1('position_1d')) #still need 1D position
position_2d = pd.read_csv(entry.fetch1('position_2d')) # need 2D position

entry = DecodeResults2D & {'nwb_file_name':nwb_copy_file_name,
                             'interval_list_name':session_name}

environment_path = entry.fetch1('classifier')
with open(environment_path, 'rb') as file:
    environment2D = pickle.load(file)

decode_path2d = entry.fetch1('posterior')
results = xr.open_zarr(decode_path2d, consolidated=False)

#classifier_path = entry.fetch1('classifier')
#with open(classifier_path, 'rb') as file:
#    classifier = pickle.load(file)

In [15]:
marks_np = np.array(marks.to_dataarray()).squeeze()

In [16]:
timestamps = np.array(position_1d.time)

### Make video for a trial

In [54]:
#log[log.change_of_mind]

In [21]:
trialInd = 7
t0 = log.loc[trialInd,'timestamp_H']
t1 = log.loc[trialInd + 1,'timestamp_H']
if np.isnan(t0):
    t0 = t1-12

t0 = 1666397811.5
t1 = 1666397813

frameToPlot = np.argwhere(np.logical_and(timestamps>=t0,timestamps<=t1)).ravel()
frame0 = frameToPlot[0]
frameLast = frameToPlot[-1]

print("t0",t0)
print("t1",t1)
print("frame0 - frameLast",frameLast - frame0)

t0 1666397811.5
t1 1666397813
frame0 - frameLast 749


In [22]:
title = "fragmented_replay"

In [ ]:
make_single_environment_movie(
    slice(frame0, frameLast),
    environment2D,
    results,
    position_2d,
    marks_np,
    movie_name=f"{nwb_copy_file_name[:13]}_session{session_name}_trial{trialInd}_t0{t0}_t1{t1}_{title}.mp4",
    sampling_frequency=1000,
    video_slowdown=8,
    position_name=["head_position_x", "head_position_y"],
    direction_name="head_orientation",
    vmax=0.07,
)

0it [00:00, ?it/s]

In [76]:
results

<xarray.Dataset> Size: 219GB
Dimensions:            (time: 904453, state: 2, x_position: 103, y_position: 98)
Coordinates:
  * state              (state) <U10 80B 'Continuous' 'Uniform'
  * time               (time) float64 7MB 1.666e+09 1.666e+09 ... 1.666e+09
  * x_position         (x_position) float64 824B 120.4 122.4 ... 321.0 323.0
  * y_position         (y_position) float64 784B 43.52 45.5 ... 233.6 235.6
Data variables:
    acausal_posterior  (time, state, x_position, y_position) float32 73GB dask.array<chunksize=(56529, 1, 7, 7), meta=np.ndarray>
    causal_posterior   (time, state, x_position, y_position) float32 73GB dask.array<chunksize=(56529, 1, 7, 7), meta=np.ndarray>
    likelihood         (time, state, x_position, y_position) float32 73GB dask.array<chunksize=(56529, 1, 7, 7), meta=np.ndarray>
Attributes:
    data_log_likelihood:  -556043.6875

In [86]:
time_slice = slice(frame0, frame0 + 20)
state_posterior = (
        results.isel(time=time_slice).acausal_posterior
        .sum(["x_position","y_position"])
    )

In [94]:
np.asarray(state_posterior.isel(time = slice(0,10)))[:,0]

array([0.9984579 , 0.9984259 , 0.99795717, 0.99761724, 0.9973561 ,
       0.9971459 , 0.9969284 , 0.9967339 , 0.99655145, 0.9962411 ],
      dtype=float32)

In [73]:
results.isel(time=[100,102]).causal_posterior.sum("state")

<xarray.DataArray 'causal_posterior' (time: 2, x_position: 103, y_position: 98)> Size: 81kB
dask.array<sum-aggregate, shape=(2, 103, 98), dtype=float32, chunksize=(2, 7, 7), chunktype=numpy.ndarray>
Coordinates:
  * time        (time) float64 16B 1.666e+09 1.666e+09
  * x_position  (x_position) float64 824B 120.4 122.4 124.4 ... 321.0 323.0
  * y_position  (y_position) float64 784B 43.52 45.5 47.48 ... 231.6 233.6 235.6